In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Demo variables
n_heads = 16
d_head = 128
n_layers = 28
n_groups = 8
group_size = n_heads // n_groups
T_max = 5000
T = 2
T_cached = 7
B = 5

Q = torch.empty([B, n_heads, T, d_head])
K = torch.empty([B, n_groups, T+T_cached, d_head])
V = torch.empty([B, n_groups, T+T_cached, d_head])

mask = torch.empty([B, 1, T, T+T_cached], dtype=torch.bool)

K_cache = torch.empty([B, n_groups, T_max, d_head])
V_cache = torch.empty([B, n_groups, T_max, d_head])

# Objective

Comprises two parts
1. GQA `repeat_interleave` (replaced by `enable_gqa=True`)
    - requires us to copy over `K` and `V` by `group_size`, as each group is the same K/V
    - in Qwen's format, (H1,H2,H3,H4) wants (K1,K1,K2,K2) or contiguous groups along the heads
2. Attention
    - the mask here, everything `True` is masked out

In [2]:
# K,V -> (B, n_heads, T+T_cached, d_head)
K_full = K.repeat_interleave(group_size, dim=1)
V_full = V.repeat_interleave(group_size, dim=1)

# K -> (B, n_heads, d_head, T+T_cached)
K_T = K_full.permute(0, 1, 3, 2)
A = (Q @ K_T) / (d_head ** 0.5)  # (B, n_heads, T, T+T_cached)
A = A.masked_fill(mask, float('-inf')) # set all noncausal attention pairs to none
A = torch.softmax(A, dim=3) @ V_full  # (B, n_heads, T+T_cached, d_head)

## Faster is FlashAttention

In [3]:
# (B, n_heads, T_max, d_head)
use_cache = True
if use_cache:
    A_ = F.scaled_dot_product_attention(
        Q, K_cache[:, :, :T_cached+T], V_cache[:, :, :T_cached+T], mask, enable_gqa=True)
else:
    A_ = F.scaled_dot_product_attention(Q, K, V, mask, enable_gqa=True) 